# Credit Risk Scoring
A bank wants to know, at application time, how likely a borrower is to become a **bad credit risk**. The two possible mistakes are not equally expensive:

- approving a bad borrower loses (part of) the loan → **cost 5**;
- rejecting a good borrower loses the interest the bank would have earned → **cost 1**.

So besides ranking applicants well (ROC-AUC), the model needs a decision threshold chosen for this **5 : 1 cost ratio**.

**Data:** UCI Statlog German Credit — 1,000 applications, 20 attributes, 30% bad risks.

In [1]:
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict, cross_val_score)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from train import CODES, COLUMNS, DATA_URL, fetch   # attribute-code table from the UCI documentation
from pathlib import Path
sns.set_theme(style='whitegrid'); SEED = 42
COST_FN, COST_FP = 5, 1

## 1. Load and decode
The raw file has no header and stores categories as codes like `A11`, `A34`. Using the dataset documentation, every code is translated into words.

In [2]:
df = pd.read_csv(fetch(DATA_URL, Path('data') / 'german.data'), sep=' ', header=None, names=COLUMNS)
print(df.shape)
print('Raw example:', df.iloc[0, :5].tolist())
for col, mapping in CODES.items():
    df[col] = df[col].map(mapping)
df['bad_risk'] = (df.pop('credit_risk') == 2).astype(int)     # raw: 1 = good, 2 = bad
assert df.notna().all().all()
df.head()

(1000, 21)
Raw example: ['A11', np.int64(6), 'A34', 'A43', np.int64(1169)]


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,bad_risk
0,< 0 DM,6,Critical account / credits elsewhere,Radio / TV,1169,Unknown / no savings,≥ 7 years,4,Male: single,None,...,Real estate,67,None,Own,2,Skilled employee / official,1,Yes,Yes,0
1,0–200 DM,48,Existing credits paid on time,Radio / TV,5951,< 100 DM,1–4 years,2,Female: divorced/separated/married,None,...,Real estate,22,None,Own,1,Skilled employee / official,1,No,Yes,1
2,No checking account,12,Critical account / credits elsewhere,Education,2096,< 100 DM,4–7 years,2,Male: single,None,...,Real estate,49,None,Own,1,Unskilled resident,2,No,Yes,0
3,< 0 DM,42,Existing credits paid on time,Furniture / equipment,7882,< 100 DM,4–7 years,2,Male: single,Guarantor,...,Savings agreement / life insurance,45,None,For free,1,Skilled employee / official,2,No,Yes,0
4,< 0 DM,24,Delays in the past,New car,4870,< 100 DM,1–4 years,3,Male: single,None,...,Unknown / none,53,None,For free,2,Skilled employee / official,2,No,Yes,1


In [3]:
df['bad_risk'].value_counts(normalize=True)

bad_risk
0    0.7
1    0.3
Name: proportion, dtype: float64

## 2. Exploratory analysis

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ['status', 'credit_history', 'savings']):
    df.groupby(col)['bad_risk'].mean().sort_values().plot.barh(ax=ax, color='#4f46e5')
    ax.set_title(f'Bad-risk rate by {col}'); ax.set_ylabel('')
plt.tight_layout()

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.kdeplot(data=df, x='duration', hue='bad_risk', common_norm=False, fill=True, ax=axes[0])
sns.kdeplot(data=df, x='amount', hue='bad_risk', common_norm=False, fill=True, ax=axes[1], log_scale=True)
plt.tight_layout()

**Findings**
- A negative checking balance (< 0 DM) is the strongest warning sign; having *no* checking account is the safest group.
- Longer and larger loans are riskier.
- Oddly, 'critical account' history has a *low* bad rate — those customers already passed a stricter screening (a classic selection effect in historic lending data).

## 3. Choosing the inputs
For the demo form I keep 13 features a loan officer would enter, and drop *sex / marital status* and *foreign-worker* status (sensitive attributes that should not drive a lending decision), plus a few weak ones. Does that cost accuracy?

In [6]:
NUM = ['duration', 'amount', 'age', 'installment_rate', 'number_credits']
CAT = ['status', 'credit_history', 'purpose', 'savings', 'employment_duration',
       'property', 'other_installment_plans', 'housing']
ALL = [c for c in df.columns if c != 'bad_risk']
y = df['bad_risk']
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

def lr_pipe(cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in cols if c not in num]
    return Pipeline([('prep', ColumnTransformer([('num', StandardScaler(), num),
                                                 ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])),
                     ('model', LogisticRegression(max_iter=2000, C=0.3))])
for name, cols in [('all 20 features', ALL), ('13 selected', NUM + CAT)]:
    print(f'{name:16s} CV ROC-AUC = {cross_val_score(lr_pipe(cols), df[cols], y, cv=cv, scoring="roc_auc").mean():.3f}')

all 20 features  CV ROC-AUC = 0.789


13 selected      CV ROC-AUC = 0.783


Almost no loss, so the smaller, fairer feature set is used from here on.

## 4. Model comparison
Train/test split 80/20 (stratified). On the training part, four models are compared with stratified 5-fold CV. The SVC is tuned with `GridSearchCV` **inside** each fold (nested CV). Its scores are turned into probabilities with Platt scaling (`CalibratedClassifierCV`, sigmoid). Tree models get raw numbers; linear ones get scaled numbers.

In [7]:
X = df[NUM + CAT]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

def prep(scale=True):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
svc_search = GridSearchCV(
    Pipeline([('prep', prep()), ('model', CalibratedClassifierCV(SVC(), method='sigmoid', cv=5, ensemble=False))]),
    [{'model__estimator__kernel': ['rbf'], 'model__estimator__C': [0.1, 0.3, 1, 3, 10],
      'model__estimator__gamma': ['scale', 0.01, 0.03]},
     {'model__estimator__kernel': ['linear'], 'model__estimator__C': [0.01, 0.1, 1]}],
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED + 1), scoring='roc_auc', n_jobs=2)
models = {
    'Logistic Regression': Pipeline([('prep', prep()), ('model', LogisticRegression(max_iter=2000, C=0.3))]),
    'SVC (tuned)': svc_search,
    'Random Forest': Pipeline([('prep', prep(False)), ('model', RandomForestClassifier(
        n_estimators=400, min_samples_leaf=3, max_features='sqrt', random_state=SEED, n_jobs=-1))]),
    'Gradient Boosting': Pipeline([('prep', prep(False)), ('model', GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=2, subsample=0.8, random_state=SEED))]),
}
rows = []
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=['roc_auc', 'f1', 'accuracy'], n_jobs=2)
    rows.append({'model': name, **{k: r[f'test_{k}'].mean() for k in ['roc_auc', 'f1', 'accuracy']},
                 'roc_auc_std': r['test_roc_auc'].std()})
pd.DataFrame(rows).sort_values('roc_auc', ascending=False).round(3)

,model,roc_auc,f1,accuracy,roc_auc_std
2,Random Forest,0.794,0.462,0.756,0.042
3,Gradient Boosting,0.784,0.500,0.746,0.038
0,Logistic Regression,0.776,0.514,0.746,0.059
1,SVC (tuned),0.775,0.496,0.755,0.050


In [8]:
svc_search.fit(X_train, y_train)
print('Best SVC on the full training set:', svc_search.best_params_, round(svc_search.best_score_, 3))

Best SVC on the full training set: {'model__estimator__C': 3, 'model__estimator__gamma': 0.03, 'model__estimator__kernel': 'rbf'} 0.777


All four models land in a narrow range (ROC-AUC ≈ 0.77–0.80, with a fold-to-fold standard deviation of 0.04–0.06); Random Forest is a little ahead and is used as the final model. Note that the default **0.5 threshold gives a low F1 for every model** — they rarely predict 'bad' — which is exactly why the threshold needs attention.

## 5. Cost-sensitive threshold
If the predicted probability *p* of being bad is calibrated, rejecting is cheaper when 5·p > 1·(1 − p), i.e. **p > 1/6 ≈ 0.17**. Random-forest probabilities are not perfectly calibrated, so I pick the threshold empirically: out-of-fold predictions on the training set, then the threshold with the lowest average cost.

In [9]:
best = models['Random Forest'].fit(X_train, y_train)
oof = cross_val_predict(best, X_train, y_train, cv=cv, method='predict_proba', n_jobs=2)[:, 1]

def avg_cost(y_true, p, t):
    tn, fp, fn, tp = confusion_matrix(y_true, (p >= t).astype(int), labels=[0, 1]).ravel()
    return (COST_FN * fn + COST_FP * fp) / len(y_true)

ts = np.round(np.arange(0.05, 0.81, 0.01), 2)
costs = [avg_cost(y_train, oof, t) for t in ts]
t_best = ts[int(np.argmin(costs))]
print(f'Cost-optimal threshold: {t_best:.2f}  (theory: {1 / 6:.2f})')
plt.figure(figsize=(7, 4)); plt.plot(ts, costs, color='#4f46e5')
plt.axvline(t_best, ls='--', color='grey'); plt.axvline(0.5, ls=':', color='grey')
plt.xlabel('Reject if P(bad) ≥ threshold'); plt.ylabel('Average cost per applicant (CV)');

Cost-optimal threshold: 0.24  (theory: 0.17)


## 6. Final evaluation on the test set

In [10]:
proba = best.predict_proba(X_test)[:, 1]
print('Test ROC-AUC:', round(roc_auc_score(y_test, proba), 3))
summary = pd.DataFrame([
    {'policy': 'approve everyone', 'avg cost': avg_cost(y_test, proba, 1.01)},
    {'policy': 'threshold 0.50', 'avg cost': avg_cost(y_test, proba, 0.5)},
    {'policy': f'threshold {t_best:.2f} (cost-optimal)', 'avg cost': avg_cost(y_test, proba, t_best)},
]).round(3)
summary

Test ROC-AUC: 0.785


,policy,avg cost
0,approve everyone,1.50
1,threshold 0.50,0.97
2,threshold 0.24 (cost-optimal),0.56


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, t in zip(axes, [0.5, t_best]):
    ConfusionMatrixDisplay.from_predictions(y_test, proba >= t, display_labels=['Good', 'Bad'],
                                            cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(f'Threshold {t:.2f}')
plt.tight_layout()
print(classification_report(y_test, proba >= t_best, target_names=['Good', 'Bad']))

              precision    recall  f1-score   support

        Good       0.89      0.52      0.66       140
         Bad       0.43      0.85      0.57        60

    accuracy                           0.62       200
   macro avg       0.66      0.69      0.62       200
weighted avg       0.75      0.62      0.63       200



At 0.5 the model lets most bad borrowers through. At the cost-optimal threshold it catches the large majority of them, at the price of rejecting more good applicants (lower precision, lower accuracy) — and the total business cost drops sharply. Accuracy is the wrong metric for this problem.

### What drives the score

In [12]:
imp = permutation_importance(best, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=SEED, n_jobs=2)
pd.Series(imp.importances_mean, index=X_test.columns).sort_values().plot.barh(
    figsize=(7, 5), color='#4f46e5', title='Permutation importance (drop in ROC-AUC)');

## 7. Conclusions
- Four model families reach a similar ROC-AUC (≈ 0.78–0.80); the 1,000-row dataset, not the algorithm, is the limit.
- Checking-account status, loan duration, credit history and amount carry most of the signal.
- **The threshold matters more than the model choice:** moving it from 0.5 to the cost-optimal value (bad loan = 5× a rejected good one) roughly halves the expected cost per applicant on the test set.
- Sensitive attributes (sex / marital status, foreign-worker status) are excluded with almost no loss in accuracy.
- Limitations: 1970s German data, small sample — a real scorecard would need recent data, calibration checks and a fairness review.

`train.py` contains the production version; the pipeline is exported to ONNX and runs in the web demo.